# Holstein-like and Jaynes–Cummings dynamics

Cleaned and organized for reproducible execution from the repository root.


##### Holstein-like and Jaynes–Cummings Models: Dissipation Sweep and Heatmap Comparison
##### 两种电子-振子耦合模型（Holstein 型与 Jaynes–Cummings 型）在开放系统（耗散）下的动力学行为对比
##### 最终输出两张热图：t=25 时刻比特的 〈σ_z〉 值对比，横轴为腔衰减 κ，纵轴为比特弛豫 γ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import (
    tensor, qeye, sigmaz, sigmam, sigmap,
    destroy, basis, mesolve
)


In [ ]:
# Base parameters
# 定义量子比特和谐振子的频率、耦合强度及谐振子尺寸
default_params = {
    'omega_q': 1.0, # qubit 能级间距
    'omega_r': 1.0, # 谐振子频率
    'g_z':     0.1, # Holstein 型耦合强度 σ_z(a + a†)
    'g_x':     0.1, # JC 型耦合强度 σ_+ a + σ_- a†
    'N':       20   # 谐振子 Fock 态截断数
}
omega_q = default_params['omega_q']
omega_r = default_params['omega_r']
g_z     = default_params['g_z']
g_x     = default_params['g_x']
N       = default_params['N']



In [ ]:
# Operators
# Qubit
# 比特：Pauli σ_z、升降算符；谐振子：破灭与产生算符
i_q = qeye(2)  # 比特单位算符
Sz  = sigmaz() # Pauli Z
Sp  = sigmap() # σ_+
Sm  = sigmam() # σ_-
# Resonator
i_r = qeye(N)   # 腔场单位算符
a   = destroy(N)# 腔场破灭算符
a_d = a.dag()   # 腔场产生算符

# Composite operators
# 合成算符到整体 Hilbert 空间
Sz_I   = tensor(Sz, i_r)      # 光子数算符
Sp_a   = tensor(Sp, a)        # σ_z ⊗ 1
Sm_ad  = tensor(Sm, a_d)      # σ_+ ⊗ a
iq_n   = tensor(i_q, a_d * a) # σ_- ⊗ a†


In [ ]:
# Hamiltonians
# 单体能项 + Holstein 耦合 + Jaynes–Cummings 耦合
H_q       = (omega_q / 2) * Sz_I        # qubit 单能项
H_r       = omega_r * iq_n              # resonator 单能项
H_hz      = g_z * tensor(Sz, a + a_d)   # σ_z(a + a†)
H_jc      = g_x * (Sp_a + Sm_ad)        # σ_+ a + σ_- a†
H_holstein = H_q + H_r + H_hz           # Holstein 总哈密顿量
H_jc_total = H_q + H_r + H_jc           # JC 总哈密顿量


In [ ]:
# Initial state and time grid
# 初始态与时间网格
tlist = np.linspace(0, 25, 200)         # 时间区间 [0,25] 分为 200 点
psi0  = tensor(basis(2,0), basis(N,0))  # 初始态：比特基态 |0>（〈σ_z〉=+1）、谐振子真空

# 耗散扫描设置
# Sweep ranges for dissipation rates
gamma_list = np.linspace(0.0, 0.1, 30)  # 比特弛豫率 γ 
kappa_list = np.linspace(0.0, 0.1, 30)  #  腔衰减率 κ


##### 比特弛豫率 γ (Qubit Relaxation Rate):描述量子比特从高能态 |1> 自发衰减至基态 |0> 的速率. 主要源于环境噪声和材料损耗，通过电磁场耦合将能量散逸到周围环境. 主要用于衡量量子比特的寿命 (T1 = 1/γ)，是评价量子处理器性能的关键指标，在量子控制与纠错中，需针对 γ 设计保护机制，如动态解耦或纠错码。
##### 腔衰减率 κ (Cavity Decay Rate):描述谐振腔（或谐振子）内光子泄漏出腔的速率。通常由腔壁材料损耗、耦合腔口泄漏或介质吸收等导致的能量散逸。主要用于决定谐振腔的品质因子 Q = ω_r/κ；Q 越高，腔内光子寿命越长。在光-物理实验中，κ 控制腔与外界的能量交换速率，影响 Purcell 效应和光子探测效率。
##### 在这里，通过扫描 γ 与 κ，我们能够：分析比特与腔在不同耗散条件下的稳定态占据。区分纯相位耦合 (Holstein) 与能量交换耦合 (JC) 在开放系统中的不同表现。简而言之， γ决定比特寿命，κ决定腔内光子寿命

In [ ]:
# Storage for final ⟨σ_z⟩
# 用数组存储最终时刻 t=25 的 〈σ_z〉 值
res_holstein = np.zeros((len(gamma_list), len(kappa_list)))
res_jc       = np.zeros_like(res_holstein)

for i, gamma in enumerate(gamma_list):
    for j, kappa in enumerate(kappa_list):
        # 构造塌缩算符
        c_ops = [np.sqrt(gamma) * tensor(Sm, i_r),  # qubit 弛豫
                 np.sqrt(kappa) * tensor(i_q, a)]   # 腔场耗散
        # 求解无向量密度矩阵：仅观测 〈σ_z〉
        sol_hz = mesolve(H_holstein, psi0, tlist, c_ops, [Sz_I])
        sol_jc = mesolve(H_jc,       psi0, tlist, c_ops, [Sz_I])
        # 保存 t=25 时刻值
        res_holstein[i, j] = sol_hz.expect[0][-1]
        res_jc[i, j]       = sol_jc.expect[0][-1]

In [ ]:
# Plotting side-by-side heatmaps
gamma_mesh, kappa_mesh = np.meshgrid(kappa_list, gamma_list)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(res_holstein, origin='lower', extent=[kappa_list[0], kappa_list[-1],
                                                              gamma_list[0], gamma_list[-1]],
                       aspect='auto')
axes[0].set(title='Holstein final ⟨σ_z⟩', xlabel='κ', ylabel='γ')
fig.colorbar(im0, ax=axes[0], label='⟨σ_z⟩ at t=25')

im1 = axes[1].imshow(res_jc, origin='lower', extent=[kappa_list[0], kappa_list[-1],
                                                        gamma_list[0], gamma_list[-1]],
                       aspect='auto')
axes[1].set(title='JC final ⟨σ_z⟩', xlabel='κ', ylabel='γ')
fig.colorbar(im1, ax=axes[1], label='⟨σ_z⟩ at t=25')

plt.tight_layout()

if __name__ == '__main__':
    plt.show()


##### 色度范围：从 +1（黄色）到 -1（深紫），对应比特由初始↑态到↓态。
##### Holstein 图：行为沿水平面（κ）平坦，说明腔衰减对 QND 耦合无影响，沿垂直方向颜色渐变，清晰反映 〈σ_z〉随 γ 指数衰减。上方（γ 大）: 〈σ_z〉 → -1，表示比特完全弛豫。下方（γ 近 0）: 〈σ_z〉 → +1，保持初始态。因此Holstein 型：QND 耦合 —— 仅随 γ 变化。
##### JC 图：横纵均有梯度，热图左下角（γ≈0,κ≈0）为黄色，右上角（γ大,κ大）为深紫，中间区域色调混合，体现两通道竞争。左侧（κ 近 0）: 仅比特弛豫，右侧（κ 大）: Purcell 效应增强，腔泄漏加速比特失激发。 因此JC 型：能量交换耦合 —— 同时受 γ 与 κ 影响。